# Macaron food photo model

This notebook trains the small model that lets the Macaron app recognise a dish from a photo, on the phone, with no internet.

**What it does**

1. Downloads two public food photo datasets.
2. Fine-tunes a small pretrained network (MobileNetV3) to tell the dishes apart.
3. Saves two files for the app: `food.onnx` (the model) and `labels.json` (the dish names).

**How to run it on Kaggle**

1. Go to kaggle.com, sign in, and choose **Create** then **New Notebook**.
2. **File** then **Import Notebook**, and upload this file.
3. In the right-hand panel, open **Session options**. Set **Accelerator** to **GPU T4 x2** and **Internet** to **On**.
4. Press **Run All** at the top. It takes roughly 30 to 60 minutes.
5. When it finishes, open the **Output** section on the right. Download `food.onnx` and `labels.json`.
6. Put both files in the `model` folder of the app's repository.

The model can only recognise the dishes listed in `MAP` below. Each entry links a dataset class to the exact dish name used in the app.

In [ ]:
!pip -q install timm datasets onnx onnxruntime
!pip -q install ddgs


In [ ]:
import json, random, numpy as np, torch, torch.nn as nn, timm
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision import transforms as T
from datasets import load_dataset
from PIL import Image

SEED = 7
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on", DEVICE)

# dataset class -> exact dish name in the app's food list
MAP_MY = {  # lowyisan/malaysian_food_images
    "fish_and_chips": "Fish and chips", "fried_noodles": "Mee goreng mamak", "fried_rice": "Nasi goreng kampung",
    "hamburger": "Cheeseburger", "kaya_toast": "Kaya toast set", "laksa": "Curry laksa",
    "mixed_rice": "Economy rice: 1 meat, 2 veg", "nasi_lemak": "Nasi lemak biasa", "popiah": "Popiah, fresh",
    "roti_canai": "Roti canai kosong", "satay": "Satay ayam",
}
MAP_F101 = {  # ethz/food101, only the classes that are common in Malaysia and exist in the app
    "sushi": "Sushi plates", "sashimi": "Salmon sashimi and rice", "ramen": "Ramen, shoyu", "bibimbap": "Bibimbap",
    "pad_thai": "Pad thai", "pizza": "Pizza", "spaghetti_bolognese": "Spaghetti bolognese",
    "spaghetti_carbonara": "Spaghetti carbonara", "caesar_salad": "Chicken caesar salad", "french_fries": "French fries",
    "ice_cream": "Ice cream", "chocolate_cake": "Chocolate cake", "cheesecake": "Cheesecake", "donuts": "Doughnut",
    "hummus": "Hummus and falafel plate", "falafel": "Hummus and falafel plate", "edamame": "Edamame",
    "miso_soup": "Miso soup", "chicken_wings": "Fried chicken", "dumplings": "Har gow",
    "omelette": "Omelette with rice and cucumber", "grilled_salmon": "Pan-seared salmon, rice and greens",
    "gyoza": "Gyoza", "takoyaki": "Takoyaki", "pho": "Pho", "spring_rolls": "Spring rolls, fried",
    "chicken_curry": "Chicken curry with rice", "peking_duck": "Roast duck rice", "waffles": "Waffle with syrup",
    "pancakes": "Pancakes with syrup", "french_toast": "French toast", "club_sandwich": "Club sandwich",
    "steak": "Steak with vegetables", "hot_dog": "Hot dog", "tiramisu": "Tiramisu", "lasagna": "Lasagna",
    "macaroni_and_cheese": "Macaroni and cheese", "garlic_bread": "Garlic bread", "samosa": "Samosa",
    "cup_cakes": "Cupcake", "frozen_yogurt": "Frozen yogurt", "tacos": "Tacos", "macarons": "Macarons",
    "fried_calamari": "Fried calamari", "greek_salad": "Garden salad", "eggs_benedict": "Eggs Benedict",
    "apple_pie": "Apple pie",
}
MAP_IN = {  # rajistics/indian_food_images (small, so it only tops up these classes)
    "chapati": "Chapati with dhal", "idli": "Idli with sambar", "masala_dosa": "Masala thosai",
    "chai": "Teh tarik", "samosa": "Samosa",
}
PER_CLASS = 500      # cap on training photos per class, to keep classes balanced
EPOCHS_HEAD, EPOCHS_ALL = 2, 8
BATCH = 64

## Load the photos

Both datasets come from the Hugging Face Hub. Food-101 is large (about 5 GB), so this step takes a few minutes.

In [ ]:
my = load_dataset("lowyisan/malaysian_food_images")
f101 = load_dataset("ethz/food101")

my_names = my["train"].features["label"].names
f_names = f101["train"].features["label"].names

try:
    india = load_dataset("rajistics/indian_food_images")
    in_names = india["train"].features["label"].names
except Exception as e:                                  # optional extra data: carry on without it
    india = None; MAP_IN = {}; print("Skipping the Indian food dataset:", e)


# ---- extra Malaysian hawker and home dishes from Wikimedia Commons ----
# Photos on Wikimedia Commons are published under licences that allow reuse, and the site offers an API for this.
# Search results are noisy (not every photo shows the dish), so expect lower accuracy for these classes.
import os, re, io, time, requests
COMMONS = {
 "ayam masak kicap": "Ayam masak kicap with rice and sayur",
 "kari ayam": "Chicken curry with rice",
 "steamed fish chinese": "Steamed fish with rice and greens",
 "kangkung belacan stir fried": "Stir-fried greens",
 "dal curry": "Dhal curry",
 "roast chicken potatoes home": "Baked chicken thigh with potatoes and salad",
 "braised tofu": "Tofu, braised",
 "tempeh goreng": "Tempeh, fried",
 "char kway teow": "Char kway teow",
 "bak kut teh": "Bak kut teh with rice",
 "hainanese chicken rice": "Chicken rice, steamed",
 "nasi briyani": "Nasi briyani ayam",
 "dosa": "Thosai with dhal and chutney",
 "chapati": "Chapati with dhal",
 "idli": "Idli with sambar",
 "wonton noodles": "Wantan mee, dry",
 "asam laksa": "Asam laksa",
 "hokkien mee kuala lumpur": "Hokkien mee (KL)",
 "mee rebus": "Mee rebus",
 "lontong": "Lontong",
 "nasi kandar": "Nasi kandar",
 "murtabak": "Murtabak ayam",
 "cendol": "Cendol",
 "ais kacang": "Ais kacang (ABC)",
 "apam balik": "Apam balik",
 "pisang goreng": "Pisang goreng",
 "curry puff": "Curry puff",
 "teh tarik": "Teh tarik",
 "rojak buah": "Rojak buah",
 "yong tau foo": "Yong tau foo, soup",
 "claypot chicken rice": "Claypot chicken rice",
 "chee cheong fun": "Chee cheong fun, plain",
 "prawn mee": "Prawn mee",
 "pan mee": "Pan mee, soup",
 "nasi dagang": "Nasi dagang",
 "soto ayam": "Soto ayam",
 "ikan bakar": "Ikan bakar with rice and ulam",
 "tandoori chicken": "Tandoori chicken with naan",
 "kuih": "Kuih, 2 pieces",
 "roti telur": "Roti telur",
 "mee siam": "Mee siam",
 "dim sum": "Dim sum breakfast",
 "char siu bao": "Char siu bao",
 "siu mai": "Siew mai",
 "lo mai gai": "Lor mai kai",
 "ayam penyet": "Nasi ayam penyet",
 "fried egg": "Fried egg",
 "boiled egg": "Boiled egg",
 "steamed white rice bowl": "White rice",
 "douhua tofu pudding": "Tau fu fah, less syrup",
 "keropok lekor": "Keropok lekor",
 "sup ayam": "Sup ayam with rice",
 "nasi lemak": "Nasi lemak biasa",
 "roti canai": "Roti canai kosong",
 "satay": "Satay ayam",
 # drinks, desserts, snacks, dim sum and pastries
 "yakult bottle": "Yakult Ace",
 "vitagen cultured milk": "Vitagen",
 "brown sugar boba milk": "Brown sugar pearl fresh milk",
 "bubble tea": "Bubble milk tea",
 "taro milk tea": "Taro milk tea with pearls",
 "iced matcha latte": "Matcha latte, iced",
 "chocolate milkshake": "Chocolate milkshake",
 "mango lassi": "Mango lassi",
 "mango smoothie": "Mango Magic smoothie",
 "berry smoothie": "All Berry Bang smoothie",
 "sugarcane juice": "Sugarcane juice",
 "kopi ais iced coffee": "Kopi ais",
 "teh ais iced milk tea": "Teh ais",
 "caramel frappuccino": "Caramel frappé",
 "iced chocolate drink": "Iced chocolate",
 "milo ais": "Milo ais",
 "watermelon juice": "Watermelon juice, no sugar",
 "caffe latte": "Latte",
 "americano coffee": "Americano",
 "cola can": "Soft drink",
 "frozen yogurt toppings": "llaollao, medium tub with 3 toppings",
 "ice cream scoop cup": "Inside Scoop, single scoop",
 "ice cream cone scoops": "Inside Scoop, double scoop in a waffle cone",
 "waffle with ice cream": "Ice cream waffle",
 "souffle pancake": "Soufflé pancakes",
 "mango bingsu": "Mango bingsu",
 "injeolmi bingsu": "Injeolmi bingsu",
 "chocolate bingsu": "Chocolate bingsu",
 "mango pomelo sago": "Mango sago pomelo",
 "red bean soup dessert": "Red bean soup",
 "bubur cha cha": "Bubur cha cha",
 "mango sticky rice": "Mango sticky rice",
 "daifuku mochi": "Mochi",
 "chocolate brownie": "Brownie",
 "chocolate chip cookie": "Chocolate chip cookie",
 "churros": "Churros with chocolate dip",
 "banana chocolate crepe": "Crêpe with banana and chocolate",
 "molten chocolate cake": "Molten lava cake with ice cream",
 "mcflurry": "McFlurry with Oreo",
 "chocolate sundae": "Chocolate sundae",
 "tiramisu": "Tiramisu",
 "cheesecake slice": "Cheesecake",
 "chocolate cake slice": "Chocolate cake",
 "macarons": "Macarons",
 "glazed doughnut": "Doughnut",
 "popcorn": "Popcorn, salted",
 "potato chips": "Potato chips",
 "chocolate bar": "Chocolate bar",
 "granola bar": "Granola bar",
 "soft serve ice cream": "Soft serve cone",
 "xiaolongbao": "Xiao long bao",
 "har gow": "Har gow",
 "steamed spare ribs dim sum": "Steamed pork ribs with black bean",
 "chicken feet dim sum": "Chicken feet in black bean sauce",
 "shrimp rice noodle roll": "Prawn cheong fun",
 "turnip cake dim sum": "Lo bak go, pan-fried radish cake",
 "taro dumpling wu gok": "Yam puff (wu kok)",
 "fried wonton": "Fried wantan",
 "shengjian mantou": "Sheng jian bao",
 "liu sha bao custard bun": "Salted egg custard bun (lau sar bao)",
 "ma lai go sponge cake": "Ma lai gao",
 "jian dui sesame ball": "Sesame balls (jin deui)",
 "char siu sou pastry": "Char siu sou",
 "mango pudding": "Mango pudding",
 "century egg congee": "Century egg and pork porridge",
 "zongzi": "Bak chang",
 "egg tart": "Egg tart",
 "fried spring rolls": "Spring rolls, fried",
 "gyoza": "Gyoza",
 "fried mantou condensed milk": "Fried mantou with condensed milk",
 "croissant": "Butter croissant",
 "almond croissant": "Almond croissant",
 "pain au chocolat": "Pain au chocolat",
 "danish pastry": "Fruit Danish",
 "cinnamon roll": "Cinnamon roll",
 "blueberry muffin": "Blueberry muffin",
 "chocolate muffin": "Chocolate muffin",
 "banana bread": "Banana bread",
 "scone jam cream": "Scone with jam and cream",
 "cream puff": "Cream puff",
 "chocolate eclair": "Chocolate éclair",
 "swiss roll cake": "Swiss roll",
 "pandan chiffon cake": "Pandan chiffon cake",
 "red velvet cake": "Red velvet cake",
 "carrot cake slice": "Carrot cake with cream cheese",
 "basque burnt cheesecake": "Burnt cheesecake",
 "mille crepe cake": "Mille crêpe",
 "fruit tart": "Fruit tart",
 "lemon tart": "Lemon tart",
 "pastel de nata": "Portuguese egg tart",
 "pineapple tart": "Pineapple tarts",
 "mooncake": "Mooncake, lotus paste",
 "taiyaki": "Taiyaki",
 "sausage roll": "Chicken sausage roll",
 "chicken pie": "Chicken pie",
 "quiche": "Quiche",
 "soft pretzel": "Pretzel, original",
 "bagel cream cheese": "Bagel with cream cheese",
 "pineapple bun bolo bao": "Polo bun with butter",
 "coffee bun rotiboy": "Mexican coffee bun",
 "sliced white bread": "White bread",
 "cupcake": "Cupcake",
 "apple pie": "Apple pie",
 "waffle syrup": "Waffle with syrup",
 "pancakes syrup": "Pancakes with syrup",
 # tong sui, bagels and trending treats
 "green bean soup dessert": "Green bean soup",
 "black sesame soup": "Black sesame paste",
 "peanut soup dessert chinese": "Peanut paste",
 "walnut soup dessert chinese": "Walnut paste",
 "almond tea dessert chinese": "Almond paste",
 "sweet potato soup ginger dessert": "Sweet potato ginger soup",
 "tangyuan": "Tang yuan in ginger syrup",
 "snow fungus soup": "Snow fungus with longan",
 "barley ginkgo dessert soup": "Barley, fu chuk and gingko",
 "bubur pulut hitam": "Bubur pulut hitam",
 "honeydew sago": "Honeydew sago",
 "leng chee kang": "Leng chee kang",
 "guilinggao": "Gui ling gao herbal jelly",
 "double skin milk": "Double-skin milk pudding",
 "taro sago dessert": "Taro sago",
 "bagel": "Bagel, plain",
 "lox bagel": "Smoked salmon and cream cheese bagel",
 "egg cheese bagel sandwich": "Egg and cheese bagel",
 "blueberry bagel": "Blueberry bagel with cream cheese",
 "shio pan salt bread": "Salt bread (shio pan)",
 "korean cream cheese garlic bread": "Korean garlic cream cheese bun",
 "dubai chocolate": "Dubai chocolate bar",
 "dubai chewy cookie": "Dubai chewy cookie",
 "dubai chocolate strawberries cup": "Dubai chocolate strawberry cup",
 "kunafa": "Kunafa",
 "cromboloni": "Cromboloni",
 "crookie croissant cookie": "Crookie",
 "croffle": "Croffle",
 "mochi donut pon de ring": "Mochi doughnut",
 "matcha tiramisu": "Matcha tiramisu",
 "banana pudding": "Banana pudding",
 "tanghulu": "Tanghulu",
 "corn cheese cup street food": "Jagung cheese",
 "fruit sando": "Fruit sando",
 "durian flesh": "Musang King durian",
 "durian cendol": "Durian cendol",
 "pandan gula melaka cake": "Pandan gula melaka cake",
 "caramel cheese popcorn": "Garrett popcorn, Chicago mix",
 "soft serve cone": "Mixue ice cream cone",
 "strawberry matcha latte": "Strawberry matcha latte",
 "pistachio latte": "Pistachio latte",
 "einspanner coffee": "Einspänner cream coffee",
 "coconut shake": "Coconut shake",
 # home-style dishes
 "fish soup tofu tomato": "Fish slice soup with tofu and tomato",
 "minced pork soup": "Minced pork soup",
 "fried fish cake": "Fried fish cake",
 "quesadilla": "Chicken quesadilla",
 # Korean BBQ and the newer dishes
 "samgyeopsal korean bbq grill": "Korean BBQ, typical meal",
 "bulgogi": "Bulgogi with rice",
 "kimchi jjigae": "Kimchi jjigae with rice",
 "sundubu jjigae": "Sundubu jjigae with rice",
 "tteokbokki": "Tteokbokki",
 "jajangmyeon": "Jajangmyeon",
 "samgyetang": "Samgyetang",
 "budae jjigae": "Army stew (budae jjigae)",
 "pajeon": "Seafood pajeon",
 "tonkatsu": "Tonkatsu set",
 "unadon": "Unagi don",
 "karaage": "Chicken karaage",
 "okonomiyaki": "Okonomiyaki",
 "yakitori": "Yakitori",
 "california roll": "California roll",
 "zaru soba": "Cold soba",
 "thai green curry": "Green curry chicken with rice",
 "som tam": "Som tam",
 "thai iced tea": "Thai milk tea",
 "char siu rice": "Char siu rice",
 "siu yuk": "Siew yuk rice",
 "sweet and sour pork": "Sweet and sour pork with rice",
 "kung pao chicken": "Kung pao chicken with rice",
 "kolo mee": "Kolo mee",
 "sarawak laksa": "Sarawak laksa",
 "nasi kerabu": "Nasi kerabu with ayam percik",
 "rendang": "Rendang daging with rice",
 "butter chicken": "Butter chicken with naan",
 "banana leaf rice": "Banana leaf rice with chicken",
 "vada": "Vadai",
 "barbecue chicken wings": "BBQ chicken wings",
 "barbecue pork ribs": "BBQ pork ribs",
 "lamb chops plate": "Lamb chop with fries",
 "full english breakfast": "Big breakfast",
 "avocado toast egg": "Avocado toast with egg",
 "big mac": "Big Mac",
 "chinese hot pot": "Hot pot, typical meal",
 "otak-otak": "Otak-otak"
}
UA = {"User-Agent": "MacaronFoodModel/1.1 (personal, non-commercial food classifier; https://github.com/SiobhanGoh/macaron)"}

def api(params, tries=6):
    """Wikimedia asks bots to go slowly. Retry with a growing pause when it says no."""
    for k in range(tries):
        try:
            r = requests.get("https://commons.wikimedia.org/w/api.php", headers=UA, timeout=30, params=params)
            if r.status_code == 200 and r.headers.get("content-type", "").startswith("application/json"):
                return r.json()
        except Exception:
            pass
        time.sleep(3 * 2 ** k)
    raise RuntimeError("Wikimedia Commons kept refusing")

def commons(term, want=120):
    out, cont = [], {}
    while len(out) < want:
        r = api({"action": "query", "format": "json", "generator": "search", "gsrsearch": f"filetype:bitmap {term}",
                 "gsrnamespace": 6, "gsrlimit": 50, "prop": "imageinfo", "iiprop": "url|mime", "iiurlwidth": 512,
                 "maxlag": 5, **cont})
        for pg in r.get("query", {}).get("pages", {}).values():
            ii = (pg.get("imageinfo") or [{}])[0]
            if ii.get("mime") in ("image/jpeg", "image/png") and ii.get("thumburl"): out.append(ii["thumburl"])
        if "continue" not in r: break
        cont = r["continue"]; time.sleep(1.0)
    return out[:want]

# ---- optional: top up each dish with photos from a public image search ----
# These are other people's photos. They are used here only to teach the model and are never shown in the app
# or shared. Keep it that way: do not publish the downloaded folder. Set WEB_PHOTOS = False to skip this.
WEB_PHOTOS = True
WEB_PER_DISH = 100

def web_photos(term, want):
    q = term + " food"
    try:
        try: from ddgs import DDGS
        except ImportError: from duckduckgo_search import DDGS
        for k in range(3):
            try: return [r["image"] for r in DDGS().images(q, max_results=want) if r.get("image")]
            except Exception as e: err = e; time.sleep(6 * (k + 1))
        raise err
    except Exception as e:
        print("  image search unavailable for", term, "-", str(e)[:80]); return []

from concurrent.futures import ThreadPoolExecutor
def grab(job):
    u, fn = job
    try:
        if not os.path.exists(fn):
            im = Image.open(io.BytesIO(requests.get(u, headers=UA, timeout=20).content)).convert("RGB")
            im.thumbnail((512, 512)); im.save(fn, quality=88)
        return fn
    except Exception: return None

slug = lambda t: "cm_" + re.sub(r"[^a-z0-9]+", "_", t.lower()).strip("_")
MAP_CM, CM_FILES = {}, {}
for term, food in COMMONS.items():
    lab = slug(term); d = f"/tmp/commons/{lab}"; os.makedirs(d, exist_ok=True); files = []   # /tmp is kept out of the saved outputs, so no photo is ever published
    try: urls = commons(term)
    except Exception as e: print("skipped", term, e); continue
    jobs = [(u, f"{d}/{i}.jpg") for i, u in enumerate(urls)]
    if WEB_PHOTOS: jobs += [(u, f"{d}/w{j}.jpg") for j, u in enumerate(web_photos(term, WEB_PER_DISH))]; time.sleep(1.0)
    time.sleep(2.0)
    with ThreadPoolExecutor(4) as pool: files = [f for f in pool.map(grab, jobs) if f]
    if len(files) >= 25: MAP_CM[lab] = food; CM_FILES[lab] = files
    print(f"{term}: {len(files)} photos" + ("" if len(files) >= 25 else "  (too few, left out)"))

CLASSES = sorted(set(MAP_MY.values()) | set(MAP_F101.values()) | set(MAP_IN.values()) | set(MAP_CM.values()))   # one class per dish in the app, so the same dish from two sources is one class
IDX = {c: i for i, c in enumerate(CLASSES)}
print(len(CLASSES), "classes")

class Files(Dataset):
    def __init__(self, rows, tf): self.rows, self.tf = rows, tf
    def __len__(self): return len(self.rows)
    def __getitem__(self, k):
        fn, y = self.rows[k]
        return self.tf(Image.open(fn).convert("RGB")), y

cm_train, cm_val = [], []
for lab, files in CM_FILES.items():
    random.Random(SEED).shuffle(files); nv = max(5, len(files) // 5)
    y = IDX[MAP_CM[lab]]; cm_val += [(fn, y) for fn in files[:nv]]; cm_train += [(fn, y) for fn in files[nv:]]
print(len(cm_train), "extra training photos from Wikimedia Commons")


def pick(ds, names, mapping, cap, seed=SEED):
    """Row indices for the wanted classes, at most `cap` per class."""
    labels = ds["label"]                                # reads the label column only, no image decoding
    by = {}
    for i, l in enumerate(labels):
        n = names[l]
        if n in mapping: by.setdefault(n, []).append(i)
    rng = random.Random(seed); out = []
    for n, idx in by.items():
        rng.shuffle(idx); out += [(i, IDX[mapping[n]]) for i in idx[:cap]]
    return out

class Photos(Dataset):
    def __init__(self, ds, rows, tf): self.ds, self.rows, self.tf = ds, rows, tf
    def __len__(self): return len(self.rows)
    def __getitem__(self, k):
        for step in range(20):   # a few photos in the public datasets are broken files: skip to the next one
            i, y = self.rows[(k + step) % len(self.rows)]
            try: return self.tf(self.ds[i]["image"].convert("RGB")), y
            except Exception: continue
        raise RuntimeError("too many unreadable photos in a row")

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

def square(img):
    """Centre square crop then resize to 224. The app prepares photos exactly this way."""
    w, h = img.size; m = min(w, h)
    return img.crop(((w - m) // 2, (h - m) // 2, (w + m) // 2, (h + m) // 2)).resize((224, 224), Image.BILINEAR)

train_tf = T.Compose([T.RandomResizedCrop(224, scale=(0.55, 1.0)), T.RandomHorizontalFlip(),
                      T.ColorJitter(0.3, 0.3, 0.3, 0.05), T.ToTensor(), T.Normalize(MEAN, STD)])
test_tf = T.Compose([T.Lambda(square), T.ToTensor(), T.Normalize(MEAN, STD)])

train_ds = ConcatDataset([Photos(my["train"], pick(my["train"], my_names, MAP_MY, PER_CLASS), train_tf),
                          Photos(f101["train"], pick(f101["train"], f_names, MAP_F101, PER_CLASS), train_tf)]
                         + ([Photos(india["train"], pick(india["train"], in_names, MAP_IN, PER_CLASS), train_tf)] if india else [])
                         + ([Files(cm_train, train_tf)] if cm_train else []))
val_ds = ConcatDataset([Photos(my["test"], pick(my["test"], my_names, MAP_MY, 150), test_tf),
                        Photos(f101["validation"], pick(f101["validation"], f_names, MAP_F101, 150), test_tf)]
                       + ([Files(cm_val, test_tf)] if cm_val else []))
print(len(train_ds), "training photos,", len(val_ds), "test photos")

train_dl = DataLoader(train_ds, BATCH, shuffle=True, num_workers=4, drop_last=True, pin_memory=True)
val_dl = DataLoader(val_ds, BATCH, shuffle=False, num_workers=4, pin_memory=True)

## Train

First only the new final layer is trained, then the whole network at a lower learning rate.

In [ ]:
model = timm.create_model("mobilenetv3_large_100", pretrained=True, num_classes=len(CLASSES)).to(DEVICE)
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
scaler = torch.cuda.amp.GradScaler(enabled=DEVICE == "cuda")

def evaluate():
    model.eval(); right = np.zeros(len(CLASSES)); seen = np.zeros(len(CLASSES))
    with torch.no_grad():
        for x, y in val_dl:
            p = model(x.to(DEVICE)).argmax(1).cpu()
            for a, b in zip(p.tolist(), y.tolist()): seen[b] += 1; right[b] += a == b
    return right.sum() / seen.sum(), right / np.maximum(seen, 1)

def run(epochs, lr, head_only):
    for n, p in model.named_parameters(): p.requires_grad = (not head_only) or n.startswith("classifier")
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=0.02)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * len(train_dl))
    for ep in range(epochs):
        model.train(); total = 0
        for x, y in train_dl:
            x, y = x.to(DEVICE), y.to(DEVICE); opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE, enabled=DEVICE == "cuda"):
                loss = loss_fn(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step(); total += loss.item()
        acc, _ = evaluate()
        print(f"epoch {ep + 1}/{epochs}  loss {total / len(train_dl):.3f}  test accuracy {acc:.1%}")

run(EPOCHS_HEAD, 2e-3, head_only=True)
run(EPOCHS_ALL, 4e-4, head_only=False)

acc, per = evaluate()
print(f"\nFinal test accuracy: {acc:.1%}\n")
for c, a in sorted(zip(CLASSES, per), key=lambda t: t[1]): print(f"{a:6.1%}  {c}")

## Save the two files for the app

In [ ]:
import onnx, onnxruntime as ort

model.eval().cpu()
dummy = torch.zeros(1, 3, 224, 224)
kw = dict(input_names=["input"], output_names=["logits"], opset_version=17)
try: torch.onnx.export(model, dummy, "food.onnx", dynamo=False, **kw)
except TypeError: torch.onnx.export(model, dummy, "food.onnx", **kw)
onnx.checker.check_model(onnx.load("food.onnx"))

import re
json.dump([{"label": re.sub(r"[^a-z0-9]+", "_", c.lower()).strip("_"), "food": c} for c in CLASSES], open("labels.json", "w"), indent=1)

# check the saved model gives the same answers as the trained one
sess = ort.InferenceSession("food.onnx")
x, y = next(iter(val_dl))
a = sess.run(None, {"input": x[:1].numpy()})[0]
b = model(x[:1]).detach().numpy()
print("largest difference between saved and trained model:", float(np.abs(a - b).max()))

import os
print(f"food.onnx: {os.path.getsize('food.onnx') / 1e6:.1f} MB.  labels.json: {len(CLASSES)} classes.")
print("Download both files from the Output panel and put them in the app's model folder.")

## Adding your own dishes later

To teach the model a dish that is not in the list:

1. Collect at least 80 to 100 of your own photos of it, taken the way you normally would.
2. Add them as another dataset class, add a line to `MAP` with the exact dish name from the app's food list, and run the notebook again.

Only use photos you took yourself or that are published under a licence that allows reuse.